In [32]:
import os
import requests
import json
import pandas as pd
import numpy as np
from bs4 import BeautifulSoup
from lxml import html
import re
from io import StringIO
import unicodedata

import undetected_chromedriver as uc
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time

pd.set_option('display.max_columns', None)
folder_path = "Datasets"

In [34]:
def click_link(driver, link, wait_sec=3):
    driver.execute_script("arguments[0].scrollIntoView();", link)
    time.sleep(wait_sec)
    driver.execute_script("arguments[0].click();", link)
    print(f"Current URL: {driver.current_url}")

def stat_get_html(driver, wait, league, season):
    index = 2026 - season
    clicks = 0
    
    link = wait.until(EC.presence_of_element_located((By.LINK_TEXT, league)))
    click_link(driver, link, wait_sec=7)
    clicks += 1
    
    link = wait.until(EC.presence_of_element_located((By.XPATH, f'//*[@id="seasons"]/tbody/tr[{index}]/td[1]/a')))
    click_link(driver, link, wait_sec=3)
    clicks += 1

    link = wait.until(EC.presence_of_element_located((By.XPATH, '//*[@id="inner_nav"]/ul/li[2]/div/ul[1]/li[1]/a')))
    click_link(driver, link, wait_sec=2)
    clicks += 1

    time.sleep(5)
    print(f"Successfully arrived! Current URL: {driver.current_url}")
    html = driver.page_source

    for _ in range(clicks):
        driver.back()
        print(f"Returned to: {driver.current_url}")
        time.sleep(1)

    return html

# season: 2025->1, 2024->2, ...
def get_stats_df(season):
    league_list = ["Premier League", "La Liga", "Ligue 1", "Fußball-Bundesliga", "Serie A"]
    final_df = None
    html_list = []
    
    driver = uc.Chrome(version_main=144)

    try:
        for league in league_list:
            driver.get("https://fbref.com/")

            wait = WebDriverWait(driver, 15)
            
            comps_link = wait.until(EC.presence_of_element_located((By.LINK_TEXT, "Competitions")))
            click_link(driver, comps_link, wait_sec=2)
            
            html = stat_get_html(driver, wait, league, season)
            html_list.append(html)
            print(f"Added html to the list, currently have {len(html_list)} htmls")

    except Exception as e:
        print(f"An error occurred: {e}")
    
    finally:
        time.sleep(5)
        driver.quit()

    if html_list:
        df_list = []
        for html in html_list:
            soup = BeautifulSoup(html, 'html')
            table = soup.find('table', id='stats_standard')
            
            if table:
                df = pd.read_html(StringIO(str(table)))[0]
                
                df.columns = ['_'.join(col).strip() if isinstance(col, tuple) else col for col in df.columns.values]
                df_list.append(df)
            else:
                print("Could not find the stats table.")
        final_df = pd.concat(df_list, ignore_index=True).rename(columns={'Unnamed: 1_level_0_Player': 'Name'})
        final_df.to_csv(os.path.join(folder_path, f"stats{season}.csv"))
    else:
        print("html not found")

    return final_df
    

In [36]:
def get_response(url):
    headers = {
        'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/144.0.0.0 Safari/537.36'
    }
    response = requests.get(url, headers=headers)

    return response

def create_url(team_name, team_id, season):
    return f"https://www.transfermarkt.us/{team_name}/startseite/verein/{team_id}/saison_id/{season}"

def parse_value(text):
    num_part = re.search(r'[0-9.]+', text).group()
    value = float(num_part)

    if 'm' in text:
        return value * 1_000_000
    elif 'k' in text:
        return value * 1_000
    elif 'bn' in text:
        return value * 1_000_000_000
    return value

def get_values_df(season):
    league_url_list = [
        "https://www.transfermarkt.us/premier-league/startseite/wettbewerb/GB1", 
        "https://www.transfermarkt.us/laliga/startseite/wettbewerb/ES1", 
        "https://www.transfermarkt.us/serie-a/startseite/wettbewerb/IT1", 
        "https://www.transfermarkt.us/bundesliga/startseite/wettbewerb/L1", 
        "https://www.transfermarkt.us/ligue-1/startseite/wettbewerb/FR1", 
    ]
    url_list = []
    
    team_url_list = []
    for league_url in league_url_list:
        print(f"going into: {league_url}")
        response = get_response(league_url)
        time.sleep(3)
    
        if response.status_code == 200:
            tree = html.fromstring(response.content) 
            rows = tree.xpath('//*[@id="yw1"]/table/tbody/tr')
            for row in rows:
                links = row.xpath('./td[2]/a/@href')
                if links:
                    full_url = "https://www.transfermarkt.us" + links[0] + ""
                    team_url_list.append(full_url)

    print(f"Found {len(team_url_list)} Teams.")
    
    team_name_id_map = {}
    
    for team_url in team_url_list:
        team_url_elements = team_url.split("/")
        team_name = team_url_elements[team_url_elements.index("www.transfermarkt.us") + 1]
        team_id = team_url_elements[team_url_elements.index("verein") + 1]
        team_name_id_map[team_name] = team_id

    for team_name in team_name_id_map:
        url_list.append(create_url(team_name, team_name_id_map[team_name], season))

    name_to_value = {}

    num = 0
    length = len(url_list)
    for url in url_list:
        # print(f"going into: {url}")
        response = get_response(url)
        s = 3
        time.sleep(s)
    
        if response.status_code == 200:
            tree = html.fromstring(response.content)
            rows = tree.xpath('//*[@id="yw1"]/table/tbody/tr')
            for row in rows: 
                name_list = row.xpath('.//td[2]//table//tr[1]//td[2]//a/text()')
                value_list = row.xpath('./td[5]/a/text()')
                if name_list and value_list:
                    name_text = name_list[0].strip()
                    value = parse_value(value_list[0].strip())
                    name_to_value[name_text] = value
        num += 1
        print(f"{num}/{length} Teams Added, Estimated Time Remaining: {(length - num) * (s + 1)}s")
    final_df = pd.DataFrame(list(name_to_value.items()), columns=['Name', 'Value (€)'])
    final_df.to_csv(os.path.join(folder_path, f"values{season}.csv"))
    return final_df

In [40]:
def normalize_name(name):
    if not isinstance(name, str): return ""
    name = unicodedata.normalize('NFD', name).encode('ascii', 'ignore').decode("utf-8")
    return name.lower().strip()

def normalize_and_merge(stats_df, name_value_df):
    stats_df['Name_Clean'] = stats_df['Name'].apply(normalize_name)
    name_value_df['Name_Clean'] = name_value_df['Name'].apply(normalize_name)

    return pd.merge(stats_df.reset_index(drop=True), 
                    name_value_df.reset_index(drop=True), 
                    on='Name_Clean', 
                    how='left')

def get_dataset(season):
    stats_df = get_stats_df(season)
    value_df = get_values_df(season)
    merged_df = normalize_and_merge(stats_df=stats_df, name_value_df=value_df)
    merged_df.to_csv(os.path.join(folder_path, f"{season}.csv"))

    return merged_df

In [ ]:
df_2021 = get_dataset(2021)

Current URL: https://fbref.com/en/comps/
Current URL: https://fbref.com/en/comps/
Current URL: https://fbref.com/en/comps/9/2021-2022/2021-2022-Premier-League-Stats
Current URL: https://fbref.com/en/comps/9/2021-2022/stats/2021-2022-Premier-League-Stats
Successfully arrived! Current URL: https://fbref.com/en/comps/9/2021-2022/stats/2021-2022-Premier-League-Stats
Returned to: https://fbref.com/en/comps/9/2021-2022/2021-2022-Premier-League-Stats
Returned to: https://fbref.com/en/comps/9/history/Premier-League-Seasons
Returned to: https://fbref.com/en/comps/
Added html to the list, currently have 1 htmls
An error occurred: Message: no such window: target window already closed
from unknown error: web view not found
  (Session info: chrome=145.0.7632.119)
Stacktrace:
0   undetected_chromedriver             0x0000000104997328 undetected_chromedriver + 6722344
1   undetected_chromedriver             0x000000010498eb8a undetected_chromedriver + 6687626
2   undetected_chromedriver             0

In [52]:
values2025_df = get_values_df(2025)

going into: https://www.transfermarkt.us/premier-league/startseite/wettbewerb/GB1
going into: https://www.transfermarkt.us/laliga/startseite/wettbewerb/ES1
going into: https://www.transfermarkt.us/serie-a/startseite/wettbewerb/IT1
going into: https://www.transfermarkt.us/bundesliga/startseite/wettbewerb/L1
going into: https://www.transfermarkt.us/ligue-1/startseite/wettbewerb/FR1
Found 96 Teams.
1/96 Teams Added, Estimated Time Remaining: 380s
2/96 Teams Added, Estimated Time Remaining: 376s
3/96 Teams Added, Estimated Time Remaining: 372s
4/96 Teams Added, Estimated Time Remaining: 368s
5/96 Teams Added, Estimated Time Remaining: 364s
6/96 Teams Added, Estimated Time Remaining: 360s
7/96 Teams Added, Estimated Time Remaining: 356s
8/96 Teams Added, Estimated Time Remaining: 352s
9/96 Teams Added, Estimated Time Remaining: 348s
10/96 Teams Added, Estimated Time Remaining: 344s
11/96 Teams Added, Estimated Time Remaining: 340s
12/96 Teams Added, Estimated Time Remaining: 336s
13/96 Team

In [103]:
values_2025 = pd.read_csv(os.path.join(folder_path, 'values2025.csv'))
values_2024 = pd.read_csv(os.path.join(folder_path, 'values2024.csv'))
values_2023 = pd.read_csv(os.path.join(folder_path, 'values2023.csv'))
values_2022 = pd.read_csv(os.path.join(folder_path, 'values2022.csv'))
values_2021 = pd.read_csv(os.path.join(folder_path, 'values2021.csv'))

stats2024 = pd.read_csv(os.path.join(folder_path, 'stats2024.csv'))
stats2023 = pd.read_csv(os.path.join(folder_path, 'stats2023.csv'))
stats2022 = pd.read_csv(os.path.join(folder_path, 'stats2022.csv'))
stats2021 = pd.read_csv(os.path.join(folder_path, 'stats2021.csv'))

# Testing :
shifted_2024_raw = normalize_and_merge(stats2024, values_2025)

# Training : 
shifted_2023_raw = normalize_and_merge(stats2023, values_2024)
shifted_2022_raw = normalize_and_merge(stats2022, values_2023)
shifted_2021_raw = normalize_and_merge(stats2021, values_2022)

# shifted_2024.to_csv(f"test_2024.csv")
# shifted_2023.to_csv(f"train_2023.csv")
# shifted_2022.to_csv(f"train_2022.csv")
# shifted_2021.to_csv(f"train_2021.csv")

In [169]:
def clean_dataframe(df):
    df = df.dropna(subset = ["Value (€)"])
    df = df.drop(columns = ["Name_Clean", 
                            "Unnamed: 0_y", 
                            "Name_y", 
                            "Unnamed: 0_x", 
                            "Unnamed: 0_level_0_Rk", 
                            "Unnamed: 24_level_0_Matches",
                           ])
    df = df.rename(columns={
        'Unnamed: 3_level_0_Pos': 'Position', 
        'Unnamed: 5_level_0_Age': 'Age',
        'Unnamed: 4_level_0_Squad': 'Squad',
        'Unnamed: 6_level_0_Born': 'Born',
        'Unnamed: 2_level_0_Nation': 'Nationality',
        'Playing Time_MP': 'Match Played',
        'Name_x': 'Name',
        'Playing Time_Starts': 'Match Started',
        'Playing Time_Min': 'Minutes Played',
        'Playing Time_90s': 'Minutes Played / 90',
        'Performance_Gls': 'Goals',
        'Performance_Ast': 'Assists',
        'Performance_G+A': 'Goals + Assists',
        'Performance_G-PK': 'Non-Penality Goals',
        'Performance_PK': 'Penalty Kick Goals',
        'Performance_PKatt': 'Penalty Kick Attempted',
        'Performance_CrdY': 'Yellow Cards',
        'Performance_CrdR': 'Red Cards',
        'Per 90 Minutes_Gls': 'Goals Per 90 Minutes',
        'Per 90 Minutes_Ast': 'Assists Per 90 Minutes',
        'Per 90 Minutes_G+A': 'G+A Per 90 Minutes',
        'Per 90 Minutes_G-PK': 'Non-Penality Goals Per 90 Minutes',
        'Per 90 Minutes_G+A-PK': 'Non-Penalty Goals + Assists/90',
        # '': '',
        # '': '',
        # '': '',
        # '': '',
        # '': '',
        # '': '',
        # '': '',
    })

    intger_column_list = ["Age", "Born", "Match Played", "Match Started", "Minutes Played", "Goals", "Assists", "Goals + Assists", 
                  "Non-Penality Goals", "Penalty Kick Goals", "Penalty Kick Attempted", "Yellow Cards", "Red Cards"]
    
    float_column_list = ["Minutes Played / 90", "Goals Per 90 Minutes", "Assists Per 90 Minutes", 
                  "G+A Per 90 Minutes", "Non-Penality Goals Per 90 Minutes", "Non-Penalty Goals + Assists/90"]

    numeric_column_list = []
    numeric_column_list.extend(intger_column_list)
    numeric_column_list.extend(float_column_list)

    for column in numeric_column_list:
        df[column] = pd.to_numeric(df[column], errors='coerce')
    
    return df

In [171]:
shifted_2024 = clean_dataframe(shifted_2024_raw)

In [173]:
shifted_2024.head()

,Name,Nationality,Position,Squad,Age,Born,Match Played,Match Started,Minutes Played,Minutes Played / 90,Goals,Assists,Goals + Assists,Non-Penality Goals,Penalty Kick Goals,Penalty Kick Attempted,Yellow Cards,Red Cards,Goals Per 90 Minutes,Assists Per 90 Minutes,G+A Per 90 Minutes,Non-Penality Goals Per 90 Minutes,Non-Penalty Goals + Assists/90,Value (€)
2,Tyler Adams,us USA,MF,Bournemouth,25,1999,28,21,1965,21.8,0,3,3,0,0,0,7,0,0.00,0.14,0.14,0.00,0.14,25000000.0
3,Tosin Adarabioyo,eng ENG,DF,Chelsea,26,1997,22,15,1409,15.7,1,1,2,1,0,0,4,0,0.06,0.06,0.13,0.06,0.13,20000000.0
4,Simon Adingra,ci CIV,MF,Brighton,22,2002,29,12,1097,12.2,2,2,4,2,0,0,0,0,0.16,0.16,0.33,0.16,0.33,22000000.0
7,Ola Aina,ng NGA,DF,Nottingham Forest,27,1996,35,35,2995,33.3,2,1,3,2,0,0,5,0,0.06,0.03,0.09,0.06,0.09,20000000.0
8,Rayan Aït-Nouri,dz ALG,"MF,DF",Wolves,23,2001,37,37,3109,34.5,4,7,11,4,0,0,7,1,0.12,0.20,0.32,0.12,0.32,40000000.0


In [175]:
shifted_2024.info()

<class 'pandas.DataFrame'>
Index: 1730 entries, 2 to 2962
Data columns (total 24 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   Name                               1730 non-null   str    
 1   Nationality                        1730 non-null   str    
 2   Position                           1730 non-null   str    
 3   Squad                              1730 non-null   str    
 4   Age                                1730 non-null   int64  
 5   Born                               1730 non-null   int64  
 6   Match Played                       1730 non-null   int64  
 7   Match Started                      1730 non-null   int64  
 8   Minutes Played                     1730 non-null   int64  
 9   Minutes Played / 90                1730 non-null   float64
 10  Goals                              1730 non-null   int64  
 11  Assists                            1730 non-null   int64  
 12  Goals + 

In [177]:
shifted_2023 = clean_dataframe(shifted_2023_raw)
shifted_2022 = clean_dataframe(shifted_2022_raw)
shifted_2021 = clean_dataframe(shifted_2021_raw)

In [187]:
shifted_2023 = shifted_2023.dropna()
shifted_2023.info()

<class 'pandas.DataFrame'>
Index: 1903 entries, 0 to 2961
Data columns (total 24 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   Name                               1903 non-null   str    
 1   Nationality                        1903 non-null   str    
 2   Position                           1903 non-null   str    
 3   Squad                              1903 non-null   str    
 4   Age                                1903 non-null   int64  
 5   Born                               1903 non-null   int64  
 6   Match Played                       1903 non-null   int64  
 7   Match Started                      1903 non-null   int64  
 8   Minutes Played                     1903 non-null   int64  
 9   Minutes Played / 90                1903 non-null   float64
 10  Goals                              1903 non-null   int64  
 11  Assists                            1903 non-null   int64  
 12  Goals + 

In [181]:
shifted_2022.info()

<class 'pandas.DataFrame'>
Index: 1881 entries, 0 to 2998
Data columns (total 24 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   Name                               1881 non-null   str    
 1   Nationality                        1881 non-null   str    
 2   Position                           1881 non-null   str    
 3   Squad                              1881 non-null   str    
 4   Age                                1881 non-null   int64  
 5   Born                               1881 non-null   int64  
 6   Match Played                       1881 non-null   int64  
 7   Match Started                      1881 non-null   int64  
 8   Minutes Played                     1881 non-null   int64  
 9   Minutes Played / 90                1881 non-null   float64
 10  Goals                              1881 non-null   int64  
 11  Assists                            1881 non-null   int64  
 12  Goals + 

In [183]:
shifted_2021.info()

<class 'pandas.DataFrame'>
Index: 1828 entries, 2 to 3035
Data columns (total 24 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   Name                               1828 non-null   str    
 1   Nationality                        1828 non-null   str    
 2   Position                           1828 non-null   str    
 3   Squad                              1828 non-null   str    
 4   Age                                1828 non-null   int64  
 5   Born                               1828 non-null   int64  
 6   Match Played                       1828 non-null   int64  
 7   Match Started                      1828 non-null   int64  
 8   Minutes Played                     1828 non-null   int64  
 9   Minutes Played / 90                1828 non-null   float64
 10  Goals                              1828 non-null   int64  
 11  Assists                            1828 non-null   int64  
 12  Goals + 

In [189]:
df_combined = pd.concat([shifted_2021, shifted_2022, shifted_2023], axis=0)

In [191]:
df_combined.info()

<class 'pandas.DataFrame'>
Index: 5612 entries, 2 to 2961
Data columns (total 24 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   Name                               5612 non-null   str    
 1   Nationality                        5612 non-null   str    
 2   Position                           5612 non-null   str    
 3   Squad                              5612 non-null   str    
 4   Age                                5612 non-null   int64  
 5   Born                               5612 non-null   int64  
 6   Match Played                       5612 non-null   int64  
 7   Match Started                      5612 non-null   int64  
 8   Minutes Played                     5612 non-null   int64  
 9   Minutes Played / 90                5612 non-null   float64
 10  Goals                              5612 non-null   int64  
 11  Assists                            5612 non-null   int64  
 12  Goals + 

In [197]:
from sklearn.model_selection import train_test_split

df_combined = df_combined.drop(columns = ["Name", "Nationality"])
df_combined = pd.get_dummies(df_combined, columns=['Position', "Squad"], dtype=int)

X, y = df_combined.drop(columns = ["Value (€)"]), df_combined["Value (€)"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [208]:
df_combined.head()

,Age,Born,Match Played,Match Started,Minutes Played,Minutes Played / 90,Goals,Assists,Goals + Assists,Non-Penality Goals,Penalty Kick Goals,Penalty Kick Attempted,Yellow Cards,Red Cards,Goals Per 90 Minutes,Assists Per 90 Minutes,G+A Per 90 Minutes,Non-Penality Goals Per 90 Minutes,Non-Penalty Goals + Assists/90,Value (€),Position_DF,"Position_DF,FW","Position_DF,MF",Position_FW,"Position_FW,MF",Position_GK,Position_MF,"Position_MF,DF","Position_MF,FW",Squad_Ajaccio,Squad_Alavés,Squad_Almería,Squad_Angers,Squad_Arminia,Squad_Arsenal,Squad_Aston Villa,Squad_Atalanta,Squad_Athletic Club,Squad_Atlético Madrid,Squad_Augsburg,Squad_Auxerre,Squad_Barcelona,Squad_Bayern Munich,Squad_Bochum,Squad_Bologna,Squad_Bordeaux,Squad_Bournemouth,Squad_Brentford,Squad_Brest,Squad_Brighton,Squad_Burnley,Squad_Cagliari,Squad_Celta Vigo,Squad_Chelsea,Squad_Clermont Foot,Squad_Cremonese,Squad_Crystal Palace,Squad_Cádiz,Squad_Darmstadt 98,Squad_Dortmund,Squad_Eintracht Frankfurt,Squad_Elche,Squad_Empoli,Squad_Espanyol,Squad_Everton,Squad_Fiorentina,Squad_Freiburg,Squad_Frosinone,Squad_Fulham,Squad_Genoa,Squad_Getafe,Squad_Girona,Squad_Gladbach,Squad_Granada,Squad_Greuther Fürth,Squad_Heidenheim,Squad_Hellas Verona,Squad_Hertha BSC,Squad_Hoffenheim,Squad_Inter,Squad_Juventus,Squad_Köln,Squad_Las Palmas,Squad_Lazio,Squad_Le Havre,Squad_Lecce,Squad_Leeds United,Squad_Leicester City,Squad_Lens,Squad_Levante,Squad_Leverkusen,Squad_Lille,Squad_Liverpool,Squad_Lorient,Squad_Luton Town,Squad_Lyon,Squad_Mainz 05,Squad_Mallorca,Squad_Manchester City,Squad_Manchester Utd,Squad_Marseille,Squad_Metz,Squad_Milan,Squad_Monaco,Squad_Montpellier,Squad_Monza,Squad_Nantes,Squad_Napoli,Squad_Newcastle United,Squad_Nice,Squad_Norwich City,Squad_Nottingham Forest,Squad_Osasuna,Squad_Paris Saint-Germain,Squad_RB Leipzig,Squad_Rayo Vallecano,Squad_Real Betis,Squad_Real Madrid,Squad_Real Sociedad,Squad_Reims,Squad_Rennes,Squad_Roma,Squad_Saint-Étienne,Squad_Salernitana,Squad_Sampdoria,Squad_Sassuolo,Squad_Schalke 04,Squad_Sevilla,Squad_Sheffield United,Squad_Southampton,Squad_Spezia,Squad_Strasbourg,Squad_Stuttgart,Squad_Torino,Squad_Tottenham Hotspur,Squad_Toulouse,Squad_Troyes,Squad_Udinese,Squad_Union Berlin,Squad_Valencia,Squad_Valladolid,Squad_Venezia,Squad_Villarreal,Squad_Watford,Squad_Werder Bremen,Squad_West Ham United,Squad_Wolfsburg,Squad_Wolves
2,20,2001,23,20,1828,20.3,1,2,3,1,0,0,4,0,0.05,0.10,0.15,0.05,0.15,22000000.0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1
3,23,1998,24,23,1995,22.2,1,3,4,1,0,0,5,0,0.05,0.14,0.18,0.05,0.18,16000000.0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,26,1995,14,10,923,10.3,2,0,2,2,0,0,0,0,0.20,0.00,0.20,0.20,0.20,42000000.0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
6,30,1991,25,17,1534,17.0,1,4,5,1,0,0,2,0,0.06,0.23,0.29,0.06,0.29,15000000.0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
7,22,1998,32,32,2853,31.7,2,12,14,2,0,0,2,0,0.06,0.38,0.44,0.06,0.44,65000000.0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,